<a href="https://colab.research.google.com/github/AkankshaB123/Optimisation/blob/main/Clustering_Geospatial%26spatial_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem Statement: Satellite Warehouse Optimization
### Context: Modeled data to minimize the overall number of large warehouses while strategically placing smaller satellite locations closer to customers, significantly reducing shipping rates.
1. Minimize Warehouses
2. And Increase satellite locations near the customers
3. So we can decrease the shipping rate

In [4]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
from scipy.spatial.distance import cdist

# ---------------------------------------------------------
# 1. Generate Synthetic Customer & Demand Data
# ---------------------------------------------------------
np.random.seed(42)
n_customers = 500

# Synthetic customer locations centered around major metro hubs (US)
hubs = [
    # North / East
    (40.7128, -74.0060),   # New York City, NY
    (42.3601, -71.0589),   # Boston, MA
    (38.9072, -77.0369),   # Washington, DC
    (41.8781, -87.6298),   # Chicago, IL
    (42.3314, -83.0458),   # Detroit, MI

    # South / Southeast
    (33.7490, -84.3880),   # Atlanta, GA
    (25.7617, -80.1918),   # Miami, FL
    (29.7604, -95.3698),   # Houston, TX
    (32.7767, -96.7970),   # Dallas, TX
    (36.1627, -86.7816),   # Nashville, TN

    # West / Southwest
    (34.0522, -118.2437),  # Los Angeles, CA
    (37.7749, -122.4194),  # San Francisco, CA
    (47.6062, -122.3321),  # Seattle, WA
    (39.7392, -104.9903),  # Denver, CO
    (33.4484, -112.0740),  # Phoenix, AZ
]
latitudes, longitudes = [], []
for _ in range(n_customers):
    hub = hubs[np.random.choice(len(hubs))]
    latitudes.append(hub[0] + np.random.normal(0, 1.5))
    longitudes.append(hub[1] + np.random.normal(0, 1.5))

df_customers = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'lat': latitudes,
    'lon': longitudes,
    'demand': np.random.randint(1, 10, size=n_customers) # Order volume
})

# ---------------------------------------------------------
# 2. Define Main vs. Satellite Warehouses
# ---------------------------------------------------------
warehouses = pd.DataFrame([
    {'name': 'Main Hub - Central', 'type': 'Main', 'lat': 39.8283, 'lon': -98.5795}, # Kansas
    {'name': 'East Satellite', 'type': 'Satellite', 'lat': 40.7128, 'lon': -74.0060}, # NY
    {'name': 'West Satellite', 'type': 'Satellite', 'lat': 34.0522, 'lon': -118.2437}, # LA
    {'name': 'Midwest Satellite', 'type': 'Satellite', 'lat': 41.8781, 'lon': -87.6298} # CHI
])

# ---------------------------------------------------------
# 3. Spatial Optimization: Calculate Distances & Allocations
# ---------------------------------------------------------
cust_coords = df_customers[['lat', 'lon']].values
wh_coords = warehouses[['lat', 'lon']].values

# Matrix of distances between every customer and warehouse (approx degrees)
distances = cdist(cust_coords, wh_coords, metric='euclidean')

# Find nearest warehouse for each customer
df_customers['nearest_wh_idx'] = np.argmin(distances, axis=1)
df_customers['assigned_wh'] = warehouses.loc[df_customers['nearest_wh_idx'], 'name'].values
df_customers['dist_to_satellite'] = np.min(distances, axis=1)

# Main warehouse distance baseline
main_wh_idx = warehouses[warehouses['type'] == 'Main'].index[0]
df_customers['dist_to_main'] = distances[:, main_wh_idx]

# Shipping cost metric (Distance * Demand * Unit Rate)
unit_rate = 15.0 # Cost multiplier per distance-demand unit
df_customers['cost_main_only'] = df_customers['dist_to_main'] * df_customers['demand'] * unit_rate
df_customers['cost_optimized'] = df_customers['dist_to_satellite'] * df_customers['demand'] * unit_rate

# Savings Summary
total_baseline = df_customers['cost_main_only'].sum()
total_optimized = df_customers['cost_optimized'].sum()
savings_pct = ((total_baseline - total_optimized) / total_baseline) * 100

print(f"Baseline Shipping Cost (Central Warehouse Only): ${total_baseline:,.2f}")
print(f"Optimized Shipping Cost (Satellite Network):     ${total_optimized:,.2f}")
print(f"Total Logistics Cost Reduction:                {savings_pct:.2f}%")

# ---------------------------------------------------------
# 4. Generate Interactive Map Visualization
# ---------------------------------------------------------
# Center map on US geographic center
m = folium.Map(location=[39.8283, -98.5795], zoom_start=4, tiles='cartodbpositron')

# Add Customer Density Heatmap
heat_data = [[row['lat'], row['lon'], row['demand']] for _, row in df_customers.iterrows()]
HeatMap(heat_data, radius=15, blur=10, max_zoom=1, name="Customer Demand Density").add_to(m)

# Plot Warehouses
for _, wh in warehouses.iterrows():
    color = 'red' if wh['type'] == 'Main' else 'blue'
    icon_shape = 'star' if wh['type'] == 'Main' else 'info-sign'

    folium.Marker(
        location=[wh['lat'], wh['lon']],
        popup=f"**{wh['name']}** ({wh['type']})",
        icon=folium.Icon(color=color, icon=icon_shape)
    ).add_to(m)

folium.LayerControl().add_to(m)

# Display map in Colab
m

Baseline Shipping Cost (Central Warehouse Only): $635,135.39
Optimized Shipping Cost (Satellite Network):     $239,281.97
Total Logistics Cost Reduction:                62.33%
